# Agente IA Básico — SIN MEMORIA

> Solo modelo + prompt. No recuerda conversaciones anteriores.

### Requisitos antes de ejecutar este notebook:

* Contar con API key de OpenAI (https://platform.openai.com)

---

## Arquitectura del proyecto

```mermaid
%% ─────────────────────────────────────────────────────────────
%% TIPO DE DIAGRAMA
%% flowchart → diagrama de flujo
%% TD        → dirección Top-Down (de arriba hacia abajo)
%% ─────────────────────────────────────────────────────────────
flowchart TD

    %% ── NODOS ────────────────────────────────────────────────
    %% Sintaxis:  ID["etiqueta visible"]
    %% El ID es interno (U, P, M, R); la etiqueta es lo que se ve
    %% ([...]) → forma de píldora/estadio — se usa para inicio y fin
    U(["Usuario\ndic: {input: 'tu pregunta'}"])

    %% ── SUBGRAPH ─────────────────────────────────────────────
    %% Agrupa nodos dentro de una caja con título
    %% subgraph ID["título"]  ...  end
    subgraph LCEL["Cadena LCEL  ·  prompt | chat"]
        direction TB  %% dirección interna del subgraph: Top-Bottom

        %% ["..."] → forma de rectángulo — representa un proceso o componente
        P["""
        ChatPromptTemplate
        langchain_core.prompts
        ──────────────
        SystemMessage → rol y comportamiento del bot
        HumanMessage  → mensaje del usuario {input}
        """]
        
        M["""
        GPT-4.1  ·  OpenAI API
        langchain.chat_models
        ──────────────
        init_chat_model('gpt-4.1', provider='openai')
        temperature = 0.7
        """]
    end

    R(["respuesta.content\nTexto final al usuario"])

    %% ── CONEXIONES ───────────────────────────────────────────
    %% Sintaxis:  A -->|"etiqueta"| B
    %% La etiqueta muestra qué tipo de dato viaja entre nodos
    U -->|"dict {input: ...}"| P
    P -->|"[SystemMessage, HumanMessage]"| M
    M -->|"AIMessage"| R

    %% ── ESTILOS ──────────────────────────────────────────────
    %% style ID  fill:#fondo, stroke:#borde, color:#texto
    %% color:#fff → texto blanco para contrastar con fondos oscuros
    style U    fill:#1976D2,stroke:#0D47A1,color:#fff
    style P    fill:#F57C00,stroke:#E65100,color:#fff
    style M    fill:#388E3C,stroke:#1B5E20,color:#fff
    style R    fill:#1976D2,stroke:#0D47A1,color:#fff
    style LCEL fill:#f3e5f5,stroke:#7B1FA2,stroke-dasharray:5 5
```

---

### Componentes utilizados

| Componente | Librería | Rol |
|---|---|---|
| `ChatPromptTemplate` | `langchain_core.prompts` | Estructura los mensajes (system + human) |
| `init_chat_model` | `langchain.chat_models` | Inicializa y configura el modelo GPT-4.1 |
| `GPT-4.1` | OpenAI API | Genera la respuesta en lenguaje natural |
| `\|` (LCEL pipe) | `langchain_core` | Encadena `prompt → chat` como pipeline |
| `chain.invoke()` | `langchain_core` | Ejecuta el pipeline con el input del usuario |

### ¿Por qué NO tiene memoria?

- Cada llamada a `chain.invoke({"input": ...})` es **independiente**
- No existe un historial de mensajes entre turnos
- El modelo no puede referenciar mensajes anteriores de la conversación

In [1]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

In [2]:
# ===========================================
# Se asegura que se carguen las variables de entorno desde el archivo .env
# ===========================================
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
openai_key = os.getenv("OPENAI_API_KEY")

if openai_key:
    print("OPENAI_API_KEY encontrada. ¡Todo listo para usar la API de OpenAI!")
else:
    raise ValueError("OPENAI_API_KEY no encontrada. Verifica tu archivo .env")

OPENAI_API_KEY encontrada. ¡Todo listo para usar la API de OpenAI!


In [4]:
# ============================================
# 1. CONFIGURACIÓN DEL MODELO
# ============================================
chat = init_chat_model(
    "gpt-4.1",
    model_provider="openai",
    temperature=0.7, # 0 a 1, no es configurable para modelos razonadores.
    api_key=openai_key,
)

In [5]:
# ============================================
# 2. PROMPT SIMPLE (Sin placeholder de historial)
# ============================================
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """Eres un asistente de IA útil y amigable llamado DataBot. 
    Responde las preguntas del usuario de manera clara y concisa.
    Responde siempre en español."""),
    ("human", "{input}")
])

In [6]:
# ============================================
# 3. CREAR LA CHAIN (Solo prompt | modelo), LCEL
# ============================================
chain = prompt | chat

## ¿Qué significa `chain = prompt | chat`?

Esta línea usa la **LCEL (LangChain Expression Language)**. El operador `|` encadena componentes de forma que la **salida de la izquierda se convierte en la entrada de la derecha**.

```
{"input": "hola"}
        ↓
      prompt          → formatea el diccionario en mensajes (SystemMessage + HumanMessage)
        ↓
[SystemMessage, HumanMessage]
        ↓
       chat           → llama al LLM con esos mensajes
        ↓
    AIMessage         ← resultado final de chain.invoke()
```

Esto funciona porque LangChain implementa el método `__or__` en cada componente. Cuando Python ve `prompt | chat`, internamente ejecuta `prompt.__or__(chat)`, que crea una secuencia encadenada llamada `RunnableSequence`.

> La cadena puede extenderse con más pasos: `prompt | chat | output_parser | otro_paso`

In [7]:
# ============================================
# 4. FUNCIÓN SIMPLE SIN MEMORIA
# ============================================
def chat_con_agente(mensaje_usuario: str) -> str:
    """
    Envía un mensaje al agente.
    ⚠️ NO mantiene historial - cada mensaje es independiente.
    """
    respuesta = chain.invoke({"input": mensaje_usuario})
    return respuesta.content

In [8]:
# ============================================
# 5. LOOP DE CONVERSACIÓN
# ============================================
def main():
    print("=" * 50)
    print("🤖 DataBot - Agente SIN MEMORIA")
    print("=" * 50)
    print("⚠️  Este agente NO recuerda mensajes anteriores")
    print("Escribe 'salir' para terminar.\n")

    print("*" * 60)
    print("💬 Comienza a chatear con DataBot:")
    
    while True:
        usuario = input("Usuario: ").strip()
        print(f"\n Persona: {usuario}\n")
        
        if usuario.lower() in ['salir', 'exit', 'quit']:
            print("\n¡Hasta luego! 👋")
            break
        
        if not usuario:
            continue
        
        respuesta = chat_con_agente(usuario)        
        print(f"🤖 DataBot: {respuesta}\n")

In [9]:
# ============================================
# 6. EJECUTAR EL AGENTE
# ============================================
main()

🤖 DataBot - Agente SIN MEMORIA
⚠️  Este agente NO recuerda mensajes anteriores
Escribe 'salir' para terminar.

************************************************************
💬 Comienza a chatear con DataBot:

 Persona: Hola, me llamo Marcos

🤖 DataBot: ¡Hola, Marcos! ¿En qué puedo ayudarte hoy?


 Persona: cómo me llamo?

🤖 DataBot: No tengo acceso a tu nombre a menos que me lo hayas dicho antes en esta conversación. ¿Quieres decirme cómo te llamas?


 Persona: salir


¡Hasta luego! 👋
